In [2]:
import math
import os
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [3]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Preparation

In [4]:
class BPETokenizer:
    def __init__(self, vocab_size=500):
        self.vocab_size = vocab_size
        self.merges = []           # list of (a, b) pairs in merge order
        self.vocab = {}            # token string -> id
        self.inv_vocab = {}        # id -> token string
        self.special = ["<pad>", "<unk>"]

    @staticmethod
    def _get_pairs(symbols):
        return Counter(zip(symbols[:-1], symbols[1:]))

    def train(self, text):
        # Start from characters; mark word boundaries with a leading '▁'
        words = re.findall(r"\S+|\s+", text)
        corpus = []
        for w in words:
            if w.isspace():
                # keep whitespace as its own char tokens so spaces are preserved
                corpus.append(list(w))
            else:
                corpus.append(["▁" + w[0]] + list(w[1:]) if w else [])

        # Initial vocab = unique chars
        charset = set(ch for seq in corpus for ch in seq)
        tokens = self.special + sorted(charset)
        token_to_id = {t: i for i, t in enumerate(tokens)}

        target_merges = self.vocab_size - len(tokens)
        for _ in range(max(0, target_merges)):
            pair_counts = Counter()
            for seq in corpus:
                for a, b in zip(seq[:-1], seq[1:]):
                    pair_counts[(a, b)] += 1
            if not pair_counts:
                break
            (a, b), _ = pair_counts.most_common(1)[0]
            new_tok = a + b
            if new_tok in token_to_id:
                break
            token_to_id[new_tok] = len(token_to_id)
            self.merges.append((a, b))
            # Apply merge across corpus
            new_corpus = []
            for seq in corpus:
                i, merged = 0, []
                while i < len(seq):
                    if i < len(seq) - 1 and seq[i] == a and seq[i + 1] == b:
                        merged.append(new_tok)
                        i += 2
                    else:
                        merged.append(seq[i])
                        i += 1
                new_corpus.append(merged)
            corpus = new_corpus
            if len(token_to_id) >= self.vocab_size:
                break

        self.vocab = token_to_id
        self.inv_vocab = {i: t for t, i in token_to_id.items()}

    def _tokenize_word(self, w, is_word=True):
        if not w:
            return []
        if is_word:
            seq = ["▁" + w[0]] + list(w[1:])
        else:
            seq = list(w)
        for a, b in self.merges:
            i, merged = 0, []
            while i < len(seq):
                if i < len(seq) - 1 and seq[i] == a and seq[i + 1] == b:
                    merged.append(a + b)
                    i += 2
                else:
                    merged.append(seq[i])
                    i += 1
            seq = merged
        return seq

    def encode(self, text):
        ids = []
        unk = self.vocab["<unk>"]
        for chunk in re.findall(r"\S+|\s+", text):
            toks = self._tokenize_word(chunk, is_word=not chunk.isspace())
            ids.extend(self.vocab.get(t, unk) for t in toks)
        return ids

    def decode(self, ids):
        pieces = [self.inv_vocab.get(i, "") for i in ids]
        text = "".join(pieces).replace("▁", "")
        return text


In [5]:
data_path = os.path.join("data", "input.txt")
with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()
print(f"Corpus length: {len(text):,} chars")

tokenizer = BPETokenizer(vocab_size=500)
tokenizer.train(text)
print(f"Vocab size: {len(tokenizer.vocab)}")

ids = tokenizer.encode(text)
print(f"Token count: {len(ids):,}")

Corpus length: 1,115,393 chars
Vocab size: 500
Token count: 671,135


In [6]:
class NextTokenDataset(Dataset):
    def __init__(self, token_ids, seq_len, stride=None):
        self.ids = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len
        self.stride = stride or seq_len // 2  # overlapping windows

    def __len__(self):
        return max(0, (len(self.ids) - self.seq_len - 1) // self.stride + 1)

    def __getitem__(self, idx):
        start = idx * self.stride
        x = self.ids[start:start + self.seq_len]
        y = self.ids[start + 1:start + 1 + self.seq_len]
        return x, y

In [7]:
seq_len = 64
split = int(0.8 * len(ids))
train_ids, val_ids = ids[:split], ids[split:]

train_ds = NextTokenDataset(train_ids, seq_len)
val_ds = NextTokenDataset(val_ids, seq_len)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
print(f"train batches {len(train_loader)} | val batches {len(val_loader)}")

train batches 263 | val batches 66


### Implement a Tiny Transformer

In [8]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * (x / rms)


def sinusoidal_positional_encoding(seq_len, dim):
    pe = torch.zeros(seq_len, dim)
    pos = torch.arange(seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, return_attn=False):
        B, T, D = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)  # (B, H, T, d)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        attn_d = self.drop(attn)
        out = attn_d @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.proj(out)
        if return_attn:
            return out, attn
        return out


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, return_attn=False):
        if return_attn:
            h, attn = self.attn(self.norm1(x), return_attn=True)
            x = x + h
            x = x + self.ff(self.norm2(x))
            return x, attn
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

In [9]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2,
                 d_ff=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_enc", sinusoidal_positional_encoding(max_seq_len, d_model))
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.norm = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, x, return_attn=False):
        T = x.size(1)
        h = self.tok_emb(x) + self.pos_enc[:T]
        h = self.drop(h)
        attns = []
        for blk in self.blocks:
            if return_attn:
                h, a = blk(h, return_attn=True)
                attns.append(a)
            else:
                h = blk(h)
        h = self.norm(h)
        logits = self.head(h)
        if return_attn:
            return logits, attns
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        max_ctx = self.pos_enc.size(0)
        for _ in range(max_new_tokens):
            ctx = idx[:, -max_ctx:]
            logits = self(ctx)[:, -1] / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [10]:
model = TinyTransformer(
    vocab_size=len(tokenizer.vocab),
    d_model=128, n_heads=4, n_layers=2, d_ff=256,
    max_seq_len=seq_len, dropout=0.1,
).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params:,}")

Model params: 327,552


In [11]:
def evaluate(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            total += loss.item() * y.numel()
            n += y.numel()
    return total / n


def train(model, train_loader, val_loader, epochs=10, lr=3e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    train_losses, val_losses = [], []
    for ep in range(1, epochs + 1):
        model.train()
        running, seen = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item() * y.numel()
            seen += y.numel()
        tr = running / seen
        va = evaluate(model, val_loader)
        train_losses.append(tr)
        val_losses.append(va)
        print(f"epoch {ep:02d} | train {tr:.4f} | val {va:.4f} | ppl {math.exp(va):.2f}")
    return train_losses, val_losses

In [12]:
train_losses, val_losses = train(model, train_loader, val_loader, epochs=10, lr=3e-4)

epoch 01 | train 13.7230 | val 5.1049 | ppl 164.83
epoch 02 | train 4.5239 | val 4.2110 | ppl 67.42
epoch 03 | train 4.0917 | val 3.9708 | ppl 53.02
epoch 04 | train 3.9101 | val 3.8320 | ppl 46.15
epoch 05 | train 3.7841 | val 3.7087 | ppl 40.80
epoch 06 | train 3.6773 | val 3.6189 | ppl 37.30
epoch 07 | train 3.5666 | val 3.5262 | ppl 33.99
epoch 08 | train 3.4531 | val 3.4500 | ppl 31.50
epoch 09 | train 3.3569 | val 3.3839 | ppl 29.49
epoch 10 | train 3.2823 | val 3.3302 | ppl 27.94


In [13]:
val_ppl = math.exp(val_losses[-1])
print(f"Final validation perplexity: {val_ppl:.2f}")

Final validation perplexity: 27.94


In [14]:
def plot_losses(train_losses, val_losses, path="loss_curve.png"):
    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label="train")
    plt.plot(val_losses, label="val")
    plt.xlabel("epoch")
    plt.ylabel("cross-entropy")
    plt.title("Training / Validation Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


plot_losses(train_losses, val_losses, "loss_curve.png")

In [15]:
def plot_val_ppl(val_losses, path="val_ppl_curve.png"):
    val_ppls = [math.exp(v) for v in val_losses]
    plt.figure(figsize=(6, 4))
    plt.plot(val_ppls, label="val ppl")
    plt.xlabel("epoch")
    plt.ylabel("perplexity")
    plt.title("Validation Perplexity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

plot_val_ppl(val_losses, "val_ppl_curve.png")

In [16]:
def plot_attention(model, tokenizer, sample_ids, path_prefix="attn"):
    model.eval()
    x = torch.tensor(sample_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

    with torch.no_grad():
        _, attns = model(x, return_attn=True)

    labels = [tokenizer.inv_vocab.get(i, "?").replace("▁", "_") for i in sample_ids]

    for li, a in enumerate(attns):
        a = a[0].cpu().numpy()  # (4, T, T)

        fig, axes = plt.subplots(2, 2, figsize=(6, 6))
        axes = axes.flatten()

        for hi in range(4):
            ax = axes[hi]
            ax.imshow(a[hi], aspect="auto", cmap="viridis")
            ax.set_title(f"L{li} H{hi}")

            ax.set_xticks(range(len(labels)))
            ax.set_yticks(range(len(labels)))
            ax.set_xticklabels(labels, rotation=90, fontsize=6)
            ax.set_yticklabels(labels, fontsize=6)

        fig.tight_layout()
        fig.savefig(f"{path_prefix}_layer{li}.png", dpi=150)
        plt.close(fig)


sample = torch.tensor(val_ids[:32], dtype=torch.long).tolist()
plot_attention(model, tokenizer, sample, path_prefix="attn")

In [17]:
prompt = "Columbia University"
prompt_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=DEVICE).unsqueeze(0)
out = model.generate(prompt_ids, max_new_tokens=200, temperature=0.9, top_k=40)
print(tokenizer.decode(out[0].tolist()))

Columbia University that with apped cousiclon.

KING RICHARD IS OF thou ED:
But sleeringving still marwer:
We you you have the tracue,
Thiobaf rate I earer not looker toud your puring the stears.

JHINGAURY a
What a Acrants so eing the of a of fabeds.

SK:
Ay, May I I Ed sle-wgaromomes cirrd wea.

HENESTER:
When I goode, Eis that I
